# Classical control v3f — does the v3 conclusion survive an independent data generator? (no neural network)

**Why this notebook exists.** In v3 the synthetic data and the inversion share the same solver and the same cold-start forcing (`u(0,t)=U0 cos(wt)` jumps to full amplitude at t=0). That is an *inverse crime*. A first check of an independent generator (RK4 in time, `GEN_R`x finer grid, face friction = mean of the two neighbouring cells, forcing at the true time, anti-aliased truth front) showed that the cold start leaves a **grid-dependent, barely damped transient** (no viscosity, weak friction): generator-vs-generator differences between successive refinements were several noise standard deviations, and coarse-solver-vs-generator differences were 3-6 noise sd. With a smooth start of the forcing (3 h cosine ramp) the same differences drop to ~0.01-0.02 (generator convergence) and ~0.2 (coarse solver vs generator) noise sd. So the transient, not the physics, was the obstacle.

**What is compared (two arms, same seeds, same observation layouts, same noise draws):**
- `same`: data from the inversion solver itself, **with the ramp** (the v3 experiment, ramped);
- `indep`: data from the independent generator (`GEN_R`x grid, RK4).

The only change to the validated v3 solver code is one line (the ramp factor on the boundary forcing). Everything else in the inversion/classification tools is copied verbatim from v3 (cell 9).

**Authoring status (honest).** The generator, its gate (volume conservation, convergence) and the discrepancy numbers were executed in a NumPy sandbox. The JAX parts are v3 code; the new glue (arm switch, build_problem for `indep`, summaries) was exercised only with a NumPy mock of JAX at tiny settings. **Cell 3 (gate) must pass before anything is interpreted.** Read `PREREG` in Cell 1 and change it *before* running if you disagree; the hash is printed and saved. Run time is not measured: run `MODE="quick"` first, then `standard`; arms/experiments can be split across sessions with `ARMS` and `RUN` (resumable).

v3 results (unramped, same solver) are **not** directly comparable with these (different forcing start); the paired `same` arm is the like-for-like baseline.

**Run order.** (1) `MODE="quick"`: Cells 1-3 must print `GATE: all required checks passed` (convergence of the generator, volume balance, JAX==NumPy, gradient check); the rest of quick mode only checks that the pipeline runs. (2) `MODE="standard"` (delete `v3f_results.json` if it exists from quick mode — the file is not resumed across modes anyway). (3) Optionally split sessions: `ARMS=["same"]` in one, `ARMS=["indep"]` in another (the verdict cell needs both). (4) Send back: the printed output of Cells 3, 6, 7 and `v3f_results.json`.

**How to read the gate output (d).** `model error` = rms(inversion solver - generator) in units of noise sd. Values well below 1 mean the two numerical solutions agree within the data noise; values above 1 mean the independent data contain a *structural* discretization error that the inversion cannot explain (chi2_ref > 1 in the `indep` arm). That is information, not failure.

In [1]:
import os, json, time, hashlib, math
import numpy as np
import jax, jax.numpy as jnp
from jax import lax
from scipy.optimize import minimize
jax.config.update("jax_enable_x64", True)

MODE = "standard"      # "quick" (pipeline check) | "standard"
TEST_MODE = False      # authoring-sandbox smoke only; leave False on Kaggle
ARMS = ["same", "indep"]
RUN = dict(GATE=True, E4F=True, E3F=True, E1F=True)
RESUME = True
GEN_R = 4              # generator refinement: 4 -> 96x48 cells, dt = 15 s, RK4 (2 was not enough for v with a sharp tilted front)
RAMP_HOURS = 3.0       # smooth start of the boundary forcing (0 = v3 cold start)

PREREG = dict(
    version="v3f-prereg-1",
    question="Do the v3 identifiability conclusions survive (i) a smooth-start forcing and (ii) data generated by an independent solver (RK4, refined grid) instead of the inversion solver?",
    arms=dict(same="data from the inversion solver itself, ramped forcing (inverse-crime baseline)", indep="data from the independent generator, ramped forcing"),
    generator_ok_if="at r=GEN_R: rms(G_r - G_2r) <= 0.25 x noise_sd for h, u and v (U0 in U0_LIST, rich regime, standard truth); volume conservation error < 1e-6",
    model_error_report="rms(inversion solver - generator) in units of noise sd, h/u/v, rich and sparse4; reported, no pass/fail",
    E4F=dict(desc="zonal 4-parameter, h-only, U0 sweep, both regimes, 10 seeds, classification exactly as v3 (E3/E4)",
             robust_if="U0* identical in the two arms for both regimes AND no U0 row differs by two class levels between arms AND at most 3 of the 10 rows change class"),
    E3F="zonal huv rows (U0,regime) = (0.15,rich),(1.0,rich),(1.0,sparse4) and the variants of v3 E3: reported in both arms, no pass/fail",
    E1F="free field, lambda path as v3 E1 (skill = 1 - RMSE/RMSE_const; recovery if skill >= 0.5 at the discrepancy-principle lambda): reported in both arms, no pass/fail",
    classification="identical to v3: identified / not_identified / optimiser_failed; IDENTIFIED if identified >= ceil(0.9*valid); NOT_IDENTIFIED if <= floor(0.1*valid); else MARGINAL; DCHI2_CRIT 13.3; tolerances 25% (Cf) and 6.25 km (boundary)",
    E5="not in this notebook", E6="not in this notebook",
)
PREREG_HASH = hashlib.sha256(json.dumps(PREREG, sort_keys=True).encode()).hexdigest()[:16]

_MODES = {
    "quick":    dict(N_STARTS=3, ZONAL_MAXITER=60,  FREE_MAXITER=120, N_SEEDS=2, N_SEEDS_E5=1,
                     L_E1=[100.0, 10.0, 1.0], U0_E4=[0.15, 0.5, 1.0]),
    "standard": dict(N_STARTS=6, ZONAL_MAXITER=120, FREE_MAXITER=250, N_SEEDS=10, N_SEEDS_E5=3,
                     L_E1=[1000.0, 300.0, 100.0, 30.0, 10.0, 3.0, 1.0], U0_E4=[0.15, 0.3, 0.5, 0.7, 1.0]),
}
CFG = dict(N_STEPS=1500, T0=150, T_STRIDE=25, U0_LIST=[0.15, 1.0], P_OBS=39,
           NOISE_FRAC=0.03, SEED=7, CF_SCALE=2.5e-3,
           TOL_BND_KM=6.25, TOL_CF_REL=0.25, DCHI2_CRIT=13.3)
CFG.update(_MODES[MODE])
if TEST_MODE:
    CFG.update(N_STEPS=60, T0=20, T_STRIDE=10, N_STARTS=2, ZONAL_MAXITER=2, FREE_MAXITER=2, N_SEEDS=2,
               L_E1=[10.0, 1.0], U0_E4=[0.5, 1.0], U0_LIST=[1.0])
REGIMES = {"rich": np.arange(CFG["T0"], CFG["N_STEPS"], CFG["T_STRIDE"]),
           "sparse4": np.linspace(CFG["N_STEPS"] // 4, CFG["N_STEPS"] - 1, 4).astype(int)}

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUT_PATH = os.path.join(OUT_DIR, "v3f_results.json" if not TEST_MODE else "v3f_results_TEST.json")
RESULTS = dict(mode=MODE, prereg=PREREG, prereg_hash=PREREG_HASH, gen_r=GEN_R, ramp_hours=RAMP_HOURS,
               cfg={k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in CFG.items()},
               gate={}, arms={a: dict(zonal_rows={}, E1F=[]) for a in ARMS}, zonal_rows={})
if RESUME and os.path.exists(OUT_PATH):
    _old = json.load(open(OUT_PATH))
    if _old.get("prereg_hash") == PREREG_HASH and _old.get("mode") == MODE and _old.get("gen_r") == GEN_R and _old.get("ramp_hours") == RAMP_HOURS:
        for a in ARMS:
            if a in _old.get("arms", {}): RESULTS["arms"][a] = _old["arms"][a]
        RESULTS["gate"] = _old.get("gate", {})
        print("RESUMED from", OUT_PATH)
    else:
        print("WARNING: existing results file has a different pre-registration hash / mode / GEN_R / ramp -> NOT resumed (kept untouched until first save).")

def _jsonable(o):
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(type(o))
def save_results():
    with open(OUT_PATH, "w") as f: json.dump({k: v for k, v in RESULTS.items() if k != "zonal_rows"}, f, default=_jsonable)

ARM = ARMS[0]
def use_arm(a):
    """select the data arm and alias RESULTS['zonal_rows'] to that arm's store (v3 tool functions write there)."""
    global ARM
    ARM = a
    RESULTS["zonal_rows"] = RESULTS["arms"][a]["zonal_rows"]

print("MODE:", MODE, "| arms:", ARMS, "| GEN_R:", GEN_R, "| ramp h:", RAMP_HOURS, "| devices:", jax.devices(), "| PRE-REGISTRATION HASH:", PREREG_HASH)
print({k: v for k, v in CFG.items()})

MODE: standard | arms: ['same', 'indep'] | GEN_R: 4 | ramp h: 3.0 | devices: [CudaDevice(id=0), CudaDevice(id=1)] | PRE-REGISTRATION HASH: 61c960b66d3ce893
{'N_STEPS': 1500, 'T0': 150, 'T_STRIDE': 25, 'U0_LIST': [0.15, 1.0], 'P_OBS': 39, 'NOISE_FRAC': 0.03, 'SEED': 7, 'CF_SCALE': 0.0025, 'TOL_BND_KM': 6.25, 'TOL_CF_REL': 0.25, 'DCHI2_CRIT': 13.3, 'N_STARTS': 6, 'ZONAL_MAXITER': 120, 'FREE_MAXITER': 250, 'N_SEEDS': 10, 'N_SEEDS_E5': 3, 'L_E1': [1000.0, 300.0, 100.0, 30.0, 10.0, 3.0, 1.0], 'U0_E4': [0.15, 0.3, 0.5, 0.7, 1.0]}


## Cell 2 — core physics + problem definition (v3 cell 3 verbatim, ONE change: smooth-start forcing `_ramp`; `build_problem` renamed `_build_problem_same`)

In [2]:
# ---- core physics + problem definition (xp-generic: numpy for truth/tests, jax.numpy for inversion) ----
LX, LY = 150e3, 50e3
NX, NY = 24, 12
DX, DY = LX / NX, LY / NY
G, H0, F0, DT = 9.81, 80.0, 6.5e-5, 60.0
OMEGA = 2 * np.pi / (12.42 * 3600.0)
RAMP_TR = RAMP_HOURS * 3600.0
def _ramp(xp, t):
    """smooth start of the boundary forcing: 0.5*(1-cos(pi t/TR)) for t < TR, then 1 (C1-continuous). RAMP_HOURS = 0 -> v3 behaviour."""
    if RAMP_TR <= 0: return 1.0
    return xp.where(t < RAMP_TR, 0.5 * (1.0 - xp.cos(xp.pi * t / RAMP_TR)), 1.0)
CF_LOW_TRUE, CF_HIGH_TRUE = 1.5e-3, 4.0e-3
B0_TRUE, S_TRUE = 0.5, 0.1          # boundary x/LX = B0 + S*(y/LY - 0.5)  (identical to the phase-0 twin)
XC = (np.arange(NX) + 0.5) * DX / LX     # normalized cell-centre coordinates
YC = (np.arange(NY) + 0.5) * DY / LY

def swe_step(xp, state, t, Cf, U0):
    eta, u, v = state
    vp = xp.pad(v, ((1, 1), (0, 0)), mode="edge"); vc = 0.5 * (vp[:, :-1] + vp[:, 1:]); v_at_u = 0.5 * (vc[:-1, :] + vc[1:, :])
    uc = 0.5 * (u[:-1, :] + u[1:, :]); up = xp.pad(uc, ((0, 0), (1, 1)), mode="edge"); u_at_v = 0.5 * (up[:, :-1] + up[:, 1:])
    Cf_u = Cf[np.clip(np.arange(NX + 1), 0, NX - 1), :]
    Cf_v = Cf[:, np.clip(np.arange(NY + 1), 0, NY - 1)]
    su = xp.sqrt(u ** 2 + v_at_u ** 2 + 1e-8); sv = xp.sqrt(u_at_v ** 2 + v ** 2 + 1e-8)
    de_dx = (eta[1:, :] - eta[:-1, :]) / DX; de_dy = (eta[:, 1:] - eta[:, :-1]) / DY
    fu = Cf_u / H0 * u * su; fv = Cf_v / H0 * v * sv
    un = u[1:-1, :] + DT * (-G * de_dx + F0 * v_at_u[1:-1, :] - fu[1:-1, :])
    u_new = xp.concatenate([xp.full((1, NY), U0 * _ramp(xp, t) * xp.cos(OMEGA * t)), un, xp.zeros((1, NY))], axis=0)
    vn = v[:, 1:-1] + DT * (-G * de_dy - F0 * u_at_v[:, 1:-1] - fv[:, 1:-1])
    v_new = xp.concatenate([xp.zeros((NX, 1)), vn, xp.zeros((NX, 1))], axis=1)
    div = (u_new[1:, :] - u_new[:-1, :]) / DX + (v_new[:, 1:] - v_new[:, :-1]) / DY
    return (eta - DT * H0 * div, u_new, v_new)

def run_obs_np(Cf, U0, n_steps, ox, oy):
    """numpy reference run -> (h,u,v) at observed cells for every step: 3 arrays (n_steps, P)."""
    s = (np.zeros((NX, NY)), np.zeros((NX + 1, NY)), np.zeros((NX, NY + 1)))
    H, U, V = [], [], []
    for n in range(n_steps):
        s = swe_step(np, s, n * DT, Cf, U0)
        e, u, v = s
        H.append(e[ox, oy]); U.append((0.5 * (u[:-1] + u[1:]))[ox, oy]); V.append((0.5 * (v[:, :-1] + v[:, 1:]))[ox, oy])
    return np.array(H), np.array(U), np.array(V)

def cf_zonal(xp, p, w_cells=0.5):
    """p = [ln Cf_low, ln Cf_high, b0, s]; smooth two-zone field on the cell grid (width = w_cells cells)."""
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    arg = (X - (p[2] + p[3] * (Y - 0.5))) / (w_cells * DX / LX)
    sig = 1.0 / (1.0 + xp.exp(-arg))
    lo, hi = xp.exp(p[0]), xp.exp(p[1])
    return lo + (hi - lo) * sig

def cf_true_field():
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    return np.where(X < B0_TRUE + S_TRUE * (Y - 0.5), CF_LOW_TRUE, CF_HIGH_TRUE)


# ---- K-zone model: K strips along x separated by K-1 parallel (common-slope) smooth boundaries ----
def cf_zonal_K(xp, theta, K, w_cells=0.5):
    """theta = [ln c_1..ln c_K, b_1..b_{K-1}, s]  (K=1: [ln c_1]).  Boundary k: x/LX = b_k + s*(y/LY-0.5)."""
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    f = xp.exp(theta[0]) * xp.ones((NX, NY))
    for k in range(K - 1):
        arg = (X - (theta[K + k] + theta[2 * K - 1] * (Y - 0.5))) / (w_cells * DX / LX)
        f = f + (xp.exp(theta[k + 1]) - xp.exp(theta[k])) / (1.0 + xp.exp(-arg))
    return f

# ---- JAX forward model returning only the observed cells (keeps memory tiny) ----
def make_forward(U0, ox, oy):
    oxj, oyj = jnp.array(ox), jnp.array(oy)
    def fwd(Cf):
        def body(state, n):
            new = swe_step(jnp, state, n * DT, Cf, U0)
            e, u, v = new
            uc = 0.5 * (u[:-1, :] + u[1:, :]); vc = 0.5 * (v[:, :-1] + v[:, 1:])
            return new, (e[oxj, oyj], uc[oxj, oyj], vc[oxj, oyj])
        init = (jnp.zeros((NX, NY)), jnp.zeros((NX + 1, NY)), jnp.zeros((NX, NY + 1)))
        _, ys = lax.scan(body, init, jnp.arange(CFG["N_STEPS"]))
        return ys
    return fwd

CF_T = cf_true_field()
BOUNDS = [(np.log(5e-4), np.log(2e-2))] * 2 + [(0.1, 0.9), (-0.5, 0.5)]
P_TRUE = np.array([np.log(CF_LOW_TRUE), np.log(CF_HIGH_TRUE), B0_TRUE, S_TRUE])

def _build_problem_same(U0, regime, variant, P_use, noise_seed=0, cf_truth=None):
    """ARM 'same': truth from the NumPy loop of the SAME scheme as the inversion solver ('inverse crime'). Layout and noise change with noise_seed."""
    cf_truth = CF_T if cf_truth is None else cf_truth
    rng = np.random.default_rng(CFG["SEED"] + noise_seed)
    cells = rng.choice(NX * NY, CFG["P_OBS"], replace=False)
    ox, oy = np.unravel_index(cells[:P_use], (NX, NY))
    t_idx = REGIMES[regime]
    Hc, Uc, Vc = run_obs_np(cf_truth, U0, CFG["N_STEPS"], ox, oy)
    clean = dict(h=Hc[t_idx], u=Uc[t_idx], v=Vc[t_idx])
    sd = {k: CFG["NOISE_FRAC"] * clean[k].std() for k in clean}
    nr = np.random.default_rng(11 + noise_seed)
    obs = {k: clean[k] + nr.normal(scale=sd[k], size=clean[k].shape) for k in clean}
    fwd = make_forward(U0, ox, oy)
    tj = jnp.array(t_idx)
    def chi2(Cf):
        H, U, V = fwd(Cf)
        c = jnp.mean(((H[tj] - obs["h"]) / sd["h"]) ** 2)
        if variant == "huv":
            c = (c + jnp.mean(((U[tj] - obs["u"]) / sd["u"]) ** 2) + jnp.mean(((V[tj] - obs["v"]) / sd["v"]) ** 2)) / 3.0
        return c
    return dict(U0=U0, regime=regime, variant=variant, P=P_use, ox=ox, oy=oy, t_idx=t_idx, obs=obs, sd=sd,
                fwd=fwd, chi2=chi2, n_obs=int(obs["h"].size * (3 if variant == "huv" else 1)))

## Cell 2b — independent generator (RK4, refined grid), truth functions, and the arm-dispatching `build_problem`

In [3]:
def _corr(a, b):
    a, b = np.asarray(a).ravel(), np.asarray(b).ravel()
    if a.std() < 1e-12 or b.std() < 1e-12: return 0.0
    return float(np.corrcoef(a, b)[0, 1])

# ======================= independent generator =======================
# Same continuous equations as the inversion solver (no advection, no viscosity, quadratic friction, f-plane, same ramp),
# DIFFERENT numerics: r-times finer grid, classical RK4 in time, truth front area-averaged on the fine cells, face friction = mean of the two neighbouring cells,
# boundary forcing evaluated at the true time. Truth Cf is evaluated at the fine cell centres (sharp step on the fine grid).
def _grid(r):
    nx, ny = NX * r, NY * r
    return nx, ny, LX / nx, LY / ny, (np.arange(nx) + 0.5) / nx, (np.arange(ny) + 0.5) / ny

def _rhs(eta, u, v, t, Cf_u, Cf_v, U0, dx, dy):
    u = u.copy(); v = v.copy()
    u[0, :] = U0 * float(_ramp(np, t)) * np.cos(OMEGA * t); u[-1, :] = 0.0
    v[:, 0] = 0.0; v[:, -1] = 0.0
    v_c = 0.5 * (v[:, :-1] + v[:, 1:]); u_c = 0.5 * (u[:-1, :] + u[1:, :])
    v_at_u = np.zeros_like(u); v_at_u[1:-1, :] = 0.5 * (v_c[:-1, :] + v_c[1:, :])
    u_at_v = np.zeros_like(v); u_at_v[:, 1:-1] = 0.5 * (u_c[:, :-1] + u_c[:, 1:])
    su = np.sqrt(u ** 2 + v_at_u ** 2 + 1e-8); sv = np.sqrt(u_at_v ** 2 + v ** 2 + 1e-8)
    du = np.zeros_like(u); dv = np.zeros_like(v)
    du[1:-1, :] = -G * (eta[1:, :] - eta[:-1, :]) / dx + F0 * v_at_u[1:-1, :] - Cf_u[1:-1, :] / H0 * u[1:-1, :] * su[1:-1, :]
    dv[:, 1:-1] = -G * (eta[:, 1:] - eta[:, :-1]) / dy - F0 * u_at_v[:, 1:-1] - Cf_v[:, 1:-1] / H0 * v[:, 1:-1] * sv[:, 1:-1]
    deta = -H0 * ((u[1:, :] - u[:-1, :]) / dx + (v[:, 1:] - v[:, :-1]) / dy)
    return deta, du, dv

def _block_mean(a, r):
    return a.reshape(NX, r, NY, r).mean(axis=(1, 3))

def run_generator(cf_fn, U0, r=2, n_steps=None, rec_volume=False):
    """(h, u, v) at COARSE cell centres, shape (n_steps, NX, NY); entry n = state at time (n+1)*DT, exactly like run_obs_np
    and the JAX forward model. Fine fields are block-averaged r x r onto the coarse cells."""
    n_steps = CFG["N_STEPS"] if n_steps is None else n_steps
    nx, ny, dx, dy, xc, yc = _grid(r)
    X, Y = np.meshgrid(xc, yc, indexing="ij")
    Cf = cf_fn(X, Y)
    Cf_u = np.zeros((nx + 1, ny)); Cf_u[1:-1, :] = 0.5 * (Cf[:-1, :] + Cf[1:, :])
    Cf_v = np.zeros((nx, ny + 1)); Cf_v[:, 1:-1] = 0.5 * (Cf[:, :-1] + Cf[:, 1:])
    eta = np.zeros((nx, ny)); u = np.zeros((nx + 1, ny)); v = np.zeros((nx, ny + 1))
    dt = DT / r
    Hh = np.empty((n_steps, NX, NY)); Uu = np.empty_like(Hh); Vv = np.empty_like(Hh); vol = np.empty(n_steps)
    f = lambda e, uu, vv, t: _rhs(e, uu, vv, t, Cf_u, Cf_v, U0, dx, dy)
    for k in range(1, n_steps * r + 1):
        t = (k - 1) * dt
        k1 = f(eta, u, v, t)
        k2 = f(eta + 0.5 * dt * k1[0], u + 0.5 * dt * k1[1], v + 0.5 * dt * k1[2], t + 0.5 * dt)
        k3 = f(eta + 0.5 * dt * k2[0], u + 0.5 * dt * k2[1], v + 0.5 * dt * k2[2], t + 0.5 * dt)
        k4 = f(eta + dt * k3[0], u + dt * k3[1], v + dt * k3[2], t + dt)
        eta = eta + dt / 6.0 * (k1[0] + 2 * k2[0] + 2 * k3[0] + k4[0])
        u = u + dt / 6.0 * (k1[1] + 2 * k2[1] + 2 * k3[1] + k4[1])
        v = v + dt / 6.0 * (k1[2] + 2 * k2[2] + 2 * k3[2] + k4[2])
        if k % r == 0:
            n = k // r - 1
            u[0, :] = U0 * float(_ramp(np, k * dt)) * np.cos(OMEGA * k * dt); u[-1, :] = 0.0   # boundary faces hold the imposed value at the new time
            Hh[n] = _block_mean(eta, r)
            Uu[n] = _block_mean(0.5 * (u[:-1, :] + u[1:, :]), r)
            Vv[n] = _block_mean(0.5 * (v[:, :-1] + v[:, 1:]), r)
            vol[n] = eta.sum() * dx * dy
    return (Hh, Uu, Vv, vol) if rec_volume else (Hh, Uu, Vv)

_B = lambda X, Y: B0_TRUE + S_TRUE * (Y - 0.5)
TRUTH_FN_SHARP = {      # point-wise definitions (the coarse 'same' arm evaluates these at coarse cell centres = v3 truths)
    "standard":   lambda X, Y: np.where(X < _B(X, Y), CF_LOW_TRUE, CF_HIGH_TRUE),
    "gradient":   lambda X, Y: CF_LOW_TRUE + (CF_HIGH_TRUE - CF_LOW_TRUE) * X,
    "uniform":    lambda X, Y: np.full_like(X, 2.5e-3),
}
def _antialias(fn, s=8):
    """area-average of fn over each generator cell (s x s sub-samples): removes the staircase of a sharp tilted front, which otherwise
    makes the (tiny) transverse velocity v converge only at first order with refinement."""
    def g(X, Y):
        nx, ny = X.shape; off = (np.arange(s) + 0.5) / s - 0.5; acc = np.zeros_like(X)
        for a in off:
            for b in off: acc += fn(X + a / nx, Y + b / ny)
        return acc / (s * s)
    return g
TRUTH_FN = {k: _antialias(f) for k, f in TRUTH_FN_SHARP.items()}
_GEN_CACHE = {}
def gen_fields(truth, U0, r=None):
    r = GEN_R if r is None else r
    key = (truth, float(U0), int(r), CFG["N_STEPS"], RAMP_HOURS)
    if key not in _GEN_CACHE: _GEN_CACHE[key] = run_generator(TRUTH_FN[truth], U0, r)
    return _GEN_CACHE[key]

def _build_problem_indep(U0, regime, variant, P_use, noise_seed=0, truth="standard"):
    """ARM 'indep': same observation layout and noise draws as v3/'same' (same rng seeds), but the clean data come from the generator."""
    rng = np.random.default_rng(CFG["SEED"] + noise_seed)
    cells = rng.choice(NX * NY, CFG["P_OBS"], replace=False)
    ox, oy = np.unravel_index(cells[:P_use], (NX, NY))
    t_idx = REGIMES[regime]
    Hf, Uf, Vf = gen_fields(truth, U0)
    clean = dict(h=Hf[t_idx][:, ox, oy], u=Uf[t_idx][:, ox, oy], v=Vf[t_idx][:, ox, oy])
    sd = {k: CFG["NOISE_FRAC"] * clean[k].std() for k in clean}
    nr = np.random.default_rng(11 + noise_seed)
    obs = {k: clean[k] + nr.normal(scale=sd[k], size=clean[k].shape) for k in clean}
    fwd = make_forward(U0, ox, oy)
    tj = jnp.array(t_idx)
    def chi2(Cf):
        H, U, V = fwd(Cf)
        c = jnp.mean(((H[tj] - obs["h"]) / sd["h"]) ** 2)
        if variant == "huv":
            c = (c + jnp.mean(((U[tj] - obs["u"]) / sd["u"]) ** 2) + jnp.mean(((V[tj] - obs["v"]) / sd["v"]) ** 2)) / 3.0
        return c
    return dict(U0=U0, regime=regime, variant=variant, P=P_use, ox=ox, oy=oy, t_idx=t_idx, obs=obs, sd=sd,
                fwd=fwd, chi2=chi2, n_obs=int(obs["h"].size * (3 if variant == "huv" else 1)))

def build_problem(U0, regime, variant, P_use, noise_seed=0, cf_truth=None):
    if ARM == "same": return _build_problem_same(U0, regime, variant, P_use, noise_seed=noise_seed, cf_truth=cf_truth)
    if cf_truth is not None: raise ValueError("arm 'indep': only the standard truth is wired in this notebook")
    return _build_problem_indep(U0, regime, variant, P_use, noise_seed=noise_seed)

## Cell 3 — GATE (generator checks + v3 solver checks with the ramp). Nothing below may be interpreted unless this passes.

In [4]:
gate = {}
_rms = lambda a: float(np.sqrt(np.mean(np.asarray(a) ** 2)))
rich = REGIMES["rich"]; sp4 = REGIMES["sparse4"]

# (a) generator: exact volume balance  V(t) = H0*LY*U0*int_0^t ramp(s) cos(w s) ds   (RK4 conserves this linear invariant to round-off)
U0g = CFG["U0_LIST"][-1]
_, _, _, vol = run_generator(TRUTH_FN["standard"], U0g, GEN_R, rec_volume=True)
sfine = np.linspace(0.0, CFG["N_STEPS"] * DT, CFG["N_STEPS"] * 200 + 1)
integrand = U0g * np.array([float(_ramp(np, s)) for s in sfine]) * np.cos(OMEGA * sfine)
cum = np.concatenate([[0.0], np.cumsum(0.5 * (integrand[1:] + integrand[:-1]) * np.diff(sfine))])
vexp = H0 * LY * cum[200::200][:CFG["N_STEPS"]]
verr = np.max(np.abs(vol - vexp)) / np.max(np.abs(vexp))
print(f"(a) generator volume balance: max rel err = {verr:.2e} ->", "PASS" if verr < 1e-6 else "FAIL"); gate["volume"] = bool(verr < 1e-6)

# (b) generator convergence: r in {1, GEN_R, 2*GEN_R}; reference = 2*GEN_R
for U0 in CFG["U0_LIST"]:
    F = {r: run_generator(TRUTH_FN["standard"], U0, r) for r in sorted({1, GEN_R, 2 * GEN_R})}
    ref = F[2 * GEN_R]
    for r in F:
        if r == 2 * GEN_R: continue
        e = [_rms(F[r][k][rich] - ref[k][rich]) / (CFG["NOISE_FRAC"] * ref[k][rich].std()) for k in range(3)]
        tag = ""
        if r == GEN_R:
            okc = bool(max(e) <= 0.25); gate[f"conv_U0={U0}"] = okc; tag = " <- GEN_R: " + ("PASS" if okc else "FAIL (use a larger GEN_R)")
        print(f"(b) U0={U0:<5} r={r}: rms(G_r - G_{2*GEN_R}) / noise_sd   h={e[0]:.3f}  u={e[1]:.3f}  v={e[2]:.3f}{tag}")

# (c) JAX solver == NumPy solver (with ramp), gradient vs finite differences, chi2 ~ 1 at the true parameters in the 'same' arm
use_arm("same")
pb = build_problem(CFG["U0_LIST"][-1], "rich", "h", CFG["P_OBS"])
H_j, U_j, V_j = [np.asarray(a) for a in pb["fwd"](jnp.array(CF_T))]
H_n, U_n, V_n = run_obs_np(CF_T, pb["U0"], CFG["N_STEPS"], pb["ox"], pb["oy"])
rel = np.max(np.abs(H_j - H_n)) / np.std(H_n)
print(f"(c1) JAX vs NumPy solver: max|dh|/std(h) = {rel:.2e} ->", "PASS" if rel < 1e-8 else "FAIL"); gate["solver_match"] = bool(rel < 1e-8)
loss_p = lambda p: pb["chi2"](cf_zonal(jnp, p))
vg = jax.jit(jax.value_and_grad(loss_p))
p0 = P_TRUE + np.array([0.2, -0.2, 0.03, -0.02])
v0, g0 = vg(jnp.array(p0)); g0 = np.asarray(g0)
fd = np.zeros(4)
for i in range(4):
    e_ = np.zeros(4); e_[i] = 1e-6
    fd[i] = (float(loss_p(jnp.array(p0 + e_))) - float(loss_p(jnp.array(p0 - e_)))) / 2e-6
err = np.max(np.abs(g0 - fd) / (np.abs(fd) + 1e-12))
print(f"(c2) grad check: max rel err = {err:.2e} ->", "PASS" if err < 1e-3 else "FAIL"); gate["grad_ok"] = bool(err < 1e-3)
c_ref = float(loss_p(jnp.array(P_TRUE)))   # note: smooth zonal model vs sharp-step truth: close to 1 but not exactly the truth
print(f"(c3) chi2 at true zonal params ('same' arm, smooth zonal model): {c_ref:.3f} (expect ~1.0 +/- {2*np.sqrt(2/pb['n_obs']):.3f}; info only)")
gcf = np.asarray(jax.grad(pb["chi2"])(jnp.array(CF_T)))
print(f"(c4) informational: max|d chi2/d Cf| = {np.abs(gcf).max():.1e}")
assert gate.get("volume") and gate.get("solver_match") and gate.get("grad_ok"), "GATE FAILED (a/c) — do not interpret anything below; send this cell's output to the assistant."
if not TEST_MODE:
    assert all(v for k, v in gate.items() if k.startswith("conv_")), "GENERATOR NOT CONVERGED at GEN_R — set GEN_R = 4 (slower) and rerun; do not interpret anything below."

# (d) size of the 'inverse crime': inversion solver vs generator at the same Cf (reported, no pass/fail)
ax, ay = np.unravel_index(np.arange(NX * NY), (NX, NY))
print("(d) model error = rms(inversion solver - generator) / noise_sd   [noise_sd = NOISE_FRAC * std(generator signal)]")
for tname in ("standard", "gradient"):
    Xc, Yc = np.meshgrid(XC, YC, indexing="ij"); Tc = TRUTH_FN_SHARP[tname](Xc, Yc)
    for U0 in CFG["U0_LIST"]:
        Hc, Uc, Vc = [a.reshape(CFG["N_STEPS"], NX, NY) for a in run_obs_np(Tc, U0, CFG["N_STEPS"], ax, ay)]
        Gf = gen_fields(tname, U0)
        for rn, ti in (("rich", rich), ("sparse4", sp4)):
            e = [_rms(a[ti] - b[ti]) / (CFG["NOISE_FRAC"] * b[ti].std()) for a, b in zip((Hc, Uc, Vc), Gf)]
            print(f"      truth={tname:9s} U0={U0:<5} {rn:8s}: h={e[0]:.2f}  u={e[1]:.2f}  v={e[2]:.2f}")
RESULTS["gate"] = {k: bool(v) for k, v in gate.items()}; save_results()
print("GATE: all required checks passed" if all(gate.values()) else "GATE: see FAIL lines above")

(a) generator volume balance: max rel err = 8.97e-11 -> PASS
(b) U0=0.15  r=1: rms(G_r - G_8) / noise_sd   h=0.095  u=0.212  v=1.136
(b) U0=0.15  r=4: rms(G_r - G_8) / noise_sd   h=0.006  u=0.013  v=0.064 <- GEN_R: PASS
(b) U0=1.0   r=1: rms(G_r - G_8) / noise_sd   h=0.060  u=0.137  v=1.901
(b) U0=1.0   r=4: rms(G_r - G_8) / noise_sd   h=0.004  u=0.009  v=0.142 <- GEN_R: PASS
(c1) JAX vs NumPy solver: max|dh|/std(h) = 7.37e-15 -> PASS
(c2) grad check: max rel err = 6.00e-08 -> PASS
(c3) chi2 at true zonal params ('same' arm, smooth zonal model): 0.998 (expect ~1.0 +/- 0.062; info only)
(c4) informational: max|d chi2/d Cf| = 2.1e-01
(d) model error = rms(inversion solver - generator) / noise_sd   [noise_sd = NOISE_FRAC * std(generator signal)]
      truth=standard  U0=0.15  rich    : h=0.19  u=0.53  v=1.52
      truth=standard  U0=0.15  sparse4 : h=1.67  u=0.35  v=2.61
      truth=standard  U0=1.0   rich    : h=0.17  u=0.45  v=5.45
      truth=standard  U0=1.0   sparse4 : h=1.63  u=0.28

## Cell 4 — shared tools (v3 cell 9 verbatim: metrics, free-field path, zonal fit, classification, resumable rows)

In [5]:
def field_metrics(cf_hat, cf_ref):
    """r, RMSE (% of the reference range) and skill = 1 - RMSE/RMSE_constant (constant predictor = oracle mean of the reference)."""
    rng_ = float(cf_ref.max() - cf_ref.min())
    rmse = float(np.sqrt(np.mean((cf_hat - cf_ref) ** 2)) / rng_)
    rmse_c = float(np.sqrt(np.mean((cf_ref - cf_ref.mean()) ** 2)) / rng_)
    return dict(r=_corr(cf_hat, cf_ref), rmse_pct=100 * rmse, skill=(1 - rmse / rmse_c) if rmse_c > 0 else float("nan"))

# ---------------- free field: lambda continuation (as v2.1, + skill, + metric reference) ----------------
def fit_free_path(pb, lams, metric_ref):
    q = np.full(NX * NY, np.log(2.5e-3)); rows, fields = [], {}
    for lam in sorted(lams, reverse=True):
        def obj(qv, lam=lam):
            lnc = qv.reshape(NX, NY)
            smooth = jnp.mean((lnc[1:, :] - lnc[:-1, :]) ** 2) + jnp.mean((lnc[:, 1:] - lnc[:, :-1]) ** 2)
            return pb["chi2"](jnp.exp(lnc)) + lam * smooth
        vgf = jax.jit(jax.value_and_grad(obj))
        def f(qv):
            v, g = vgf(jnp.array(qv)); return float(v), np.asarray(g, dtype=np.float64)
        res = minimize(f, q, jac=True, method="L-BFGS-B", bounds=[(np.log(5e-4), np.log(2e-2))] * (NX * NY),
                       options=dict(maxiter=CFG["FREE_MAXITER"], ftol=1e-12, gtol=1e-8))
        q = res.x; cf = np.exp(q).reshape(NX, NY)
        m = field_metrics(cf, metric_ref)
        rows.append(dict(lam=float(lam), **m, chi2_data=float(pb["chi2"](jnp.array(cf))), cf_std=float(cf.std()), cf_mean=float(cf.mean()),
                         nit=int(res.nit), conv=bool(res.nit < CFG["FREE_MAXITER"])))
        fields[float(lam)] = cf
    tol = 1 + 2 * np.sqrt(2 / pb["n_obs"])
    ok = [i for i, r_ in enumerate(rows) if r_["chi2_data"] <= tol]
    dp = ok[0] if ok else len(rows) - 1                    # rows descend in lambda: first = largest lambda at noise level
    oracle = int(np.nanargmax([r_["skill"] if np.isfinite(r_["skill"]) else -9 for r_ in rows]))
    return rows, dp, oracle, fields

def print_free(tag, rows, dp, oracle):
    print(tag)
    for i, r_ in enumerate(rows):
        t = ("DP " if i == dp else "   ") + ("ORC" if i == oracle else "   ")
        print(f"        {t} lam={r_['lam']:7.1f}  r={r_['r']:6.3f}  RMSE={r_['rmse_pct']:6.1f}%  skill={r_['skill']:6.3f}  chi2={r_['chi2_data']:.3f}  std={r_['cf_std']:.5f}  nit={r_['nit']}  conv={r_['conv']}")

# ---------------- zonal 4-parameter (v2.1 fit + edge error) ----------------
def fit_zonal(pb, p0):
    vgf = jax.jit(jax.value_and_grad(lambda p: pb["chi2"](cf_zonal(jnp, p))))
    def f(p):
        v, g = vgf(jnp.array(p)); return float(v), np.asarray(g, dtype=np.float64)
    res = minimize(f, p0, jac=True, method="L-BFGS-B", bounds=BOUNDS, options=dict(maxiter=CFG["ZONAL_MAXITER"], ftol=1e-12, gtol=1e-8))
    p = res.x
    off = float((p[2] - B0_TRUE) * LX / 1e3); tilt = float((p[3] - S_TRUE) * LX / 1e3)
    return dict(cf_low=float(np.exp(p[0])), cf_high=float(np.exp(p[1])), b0=float(p[2]), s=float(p[3]), chi2=float(res.fun), nit=int(res.nit),
                bnd_km=float(np.mean(np.abs((p[2] - B0_TRUE) + (p[3] - S_TRUE) * (YC - 0.5))) * LX / 1e3),
                off_km=off, tilt_km=tilt, edge_km=float(max(abs(off - tilt / 2), abs(off + tilt / 2))))

def zonal_one_seed(U0, regime, variant, P_use, ns, cf_truth=None):
    pbz = build_problem(U0, regime, variant, P_use, noise_seed=ns, cf_truth=cf_truth)
    chi2_ref = float(pbz["chi2"](cf_zonal(jnp, jnp.array(P_TRUE))))
    sols = []
    for k in range(CFG["N_STARTS"]):
        r = np.random.default_rng(100 + k + 17 * ns)
        p0 = np.array([np.log(2.5e-3) + r.normal(0, 0.4), np.log(2.5e-3) + r.normal(0, 0.4), r.uniform(0.3, 0.7), r.uniform(-0.2, 0.2)])
        sols.append(fit_zonal(pbz, p0))
    return dict(noise_seed=ns, chi2_ref=chi2_ref, n_obs=pbz["n_obs"], sols=sols)

def classify_seed(sd, tol_cf=0.25, tol_bnd=6.25, use_edge=False):
    n = sd["n_obs"]; sols = sd["sols"]; best = min(s["chi2"] for s in sols)
    if best > sd["chi2_ref"] + 2 * np.sqrt(2 / n): return "optimiser_failed"
    equiv = [s for s in sols if n * (s["chi2"] - best) <= CFG["DCHI2_CRIT"]]
    key = "edge_km" if use_edge else "bnd_km"
    ok = all(s[key] <= tol_bnd and abs(s["cf_low"] / CF_LOW_TRUE - 1) <= tol_cf and abs(s["cf_high"] / CF_HIGH_TRUE - 1) <= tol_cf for s in equiv)
    return "identified" if ok else "not_identified"

def misspec_flag(sd): return sd["chi2_ref"] > 1 + 5 * np.sqrt(2 / sd["n_obs"])

def classify_row(per_seed, tol_cf=0.25, tol_bnd=6.25, use_edge=False, exclude_flagged=False):
    seeds = [sd for sd in per_seed if not (exclude_flagged and misspec_flag(sd))]
    labs = [classify_seed(sd, tol_cf, tol_bnd, use_edge) for sd in seeds]
    ni, nn, nf = labs.count("identified"), labs.count("not_identified"), labs.count("optimiser_failed")
    valid = ni + nn
    if valid < 0.8 * len(seeds) or valid == 0: cls = "INVALID"
    elif ni >= math.ceil(0.9 * valid): cls = "IDENTIFIED"
    elif ni <= math.floor(0.1 * valid): cls = "NOT_IDENTIFIED"
    else: cls = "MARGINAL"
    return dict(identified=ni, not_identified=nn, optimiser_failed=nf, n_seeds=len(seeds), cls=cls)

def run_zonal_row(U0, regime, variant, n_seeds):
    key = f"U0={U0}|{regime}|{variant}"
    rows = RESULTS["zonal_rows"].setdefault(key, [])
    P_use = CFG["P_OBS"] if variant == "h" else CFG["P_OBS"] // 3
    t0 = time.time()
    for ns in range(len(rows), n_seeds):
        rows.append(zonal_one_seed(U0, regime, variant, P_use, ns)); save_results()
    c = classify_row(rows)
    print(f"[zonal] {key:28s} seeds={len(rows)}  identified {c['identified']}  not-id {c['not_identified']}  opt-failed {c['optimiser_failed']}  -> {c['cls']}   "
          f"misspec-flagged seeds: {sum(misspec_flag(s) for s in rows)}   [{time.time()-t0:.0f}s this call]")
    return rows

## Cell 5 — E4F / E3F: zonal rows in both arms (h-only U0 sweep; huv rows). Resumable; one JSON store per arm.

In [6]:
t0 = time.time()
for arm in ARMS:
    use_arm(arm)
    if RUN["E4F"]:
        for reg in REGIMES:
            for U0 in CFG["U0_E4"]:
                run_zonal_row(U0, reg, "h", CFG["N_SEEDS"])
    if RUN["E3F"]:
        for (U0, reg) in [(0.15, "rich"), (1.0, "rich"), (1.0, "sparse4")]:
            if U0 in CFG["U0_LIST"]: run_zonal_row(U0, reg, "huv", CFG["N_SEEDS"])
    print(f"--- arm {arm} done [{time.time()-t0:.0f}s]")
save_results()

[zonal] U0=0.15|rich|h               seeds=10  identified 0  not-id 10  opt-failed 0  -> NOT_IDENTIFIED   misspec-flagged seeds: 0   [575s this call]
[zonal] U0=0.3|rich|h                seeds=10  identified 0  not-id 10  opt-failed 0  -> NOT_IDENTIFIED   misspec-flagged seeds: 0   [583s this call]
[zonal] U0=0.5|rich|h                seeds=10  identified 0  not-id 10  opt-failed 0  -> NOT_IDENTIFIED   misspec-flagged seeds: 0   [716s this call]
[zonal] U0=0.7|rich|h                seeds=10  identified 0  not-id 10  opt-failed 0  -> NOT_IDENTIFIED   misspec-flagged seeds: 0   [611s this call]
[zonal] U0=1.0|rich|h                seeds=10  identified 0  not-id 10  opt-failed 0  -> NOT_IDENTIFIED   misspec-flagged seeds: 0   [581s this call]
[zonal] U0=0.15|sparse4|h            seeds=10  identified 0  not-id 10  opt-failed 0  -> NOT_IDENTIFIED   misspec-flagged seeds: 0   [574s this call]
[zonal] U0=0.3|sparse4|h             seeds=10  identified 0  not-id 10  opt-failed 0  -> NOT_IDENTIF

## Cell 6 — summaries and the pre-registered robustness verdict (E4F)

In [7]:
LV = {"NOT_IDENTIFIED": 0, "MARGINAL": 1, "IDENTIFIED": 2}
def _row(arm, U0, reg, var="h"):
    return RESULTS["arms"][arm]["zonal_rows"].get(f"U0={U0}|{reg}|{var}")
def _ustar(arm, reg):
    tab = []
    for U0 in sorted(CFG["U0_E4"]):
        rows = _row(arm, U0, reg)
        if rows: tab.append((U0, classify_row(rows)["cls"]))
    for i, (u, c) in enumerate(tab):
        if all(cc == "IDENTIFIED" for _, cc in tab[i:]): return u
    return None

print("=== E4F: zonal h-only, class per U0 (identified/not-id/opt-failed -> class), chi2_ref = chi2 at the true zonal parameters (model-error indicator) ===")
changed, twolevel, ustars, n_rows = 0, False, {}, 0
for reg in REGIMES:
    print(f"[{reg}]")
    for U0 in sorted(CFG["U0_E4"]):
        parts, cls = [], {}
        for arm in ARMS:
            rows = _row(arm, U0, reg)
            if not rows: parts.append(f"{arm}: (missing)"); continue
            c = classify_row(rows); cls[arm] = c["cls"]
            parts.append(f"{arm}: {c['identified']}/{c['not_identified']}/{c['optimiser_failed']} {c['cls']:15s} chi2_ref={np.mean([s['chi2_ref'] for s in rows]):.2f}")
        print(f"   U0={U0:<5} " + " | ".join(parts))
        if len(cls) == 2:
            n_rows += 1
            if cls[ARMS[0]] != cls[ARMS[1]]: changed += 1
            if abs(LV[cls[ARMS[0]]] - LV[cls[ARMS[1]]]) >= 2: twolevel = True
    ustars[reg] = {arm: _ustar(arm, reg) for arm in ARMS}
    print(f"   U0* per arm: {ustars[reg]}")
if len(ARMS) == 2 and n_rows:
    same_u = all(ustars[r][ARMS[0]] == ustars[r][ARMS[1]] for r in REGIMES)
    robust = same_u and (not twolevel) and (changed <= 3)
    print(f"\nPRE-REGISTERED VERDICT (E4F): U0* identical in both arms: {same_u} | any two-level difference: {twolevel} | rows changing class: {changed}/{n_rows} (limit 3)")
    print("   =>", "ROBUST" if robust else "NOT ROBUST (report which rows differ; v3 conclusion must be qualified)")
    RESULTS["E4F_verdict"] = dict(same_u=bool(same_u), twolevel=bool(twolevel), changed=int(changed), n_rows=int(n_rows), robust=bool(robust), ustars=ustars)

print("\n=== E3F: zonal rows and v3 variants, both arms (id/not-id/opt-fail -> class) ===")
VARIANTS = [("PRIMARY: Cf 25%, mean-bnd", dict(tol_cf=0.25, use_edge=False)),
            ("Cf 15%, mean-bnd", dict(tol_cf=0.15, use_edge=False)),
            ("Cf 35%, mean-bnd", dict(tol_cf=0.35, use_edge=False)),
            ("Cf 25%, EDGE-bnd", dict(tol_cf=0.25, use_edge=True))]
for arm in ARMS:
    print(f"[arm {arm}]")
    for key, per_seed in RESULTS["arms"][arm]["zonal_rows"].items():
        if not key.endswith("|huv"): continue
        print(f"  {key}  (seeds={len(per_seed)}, misspec-flagged={sum(misspec_flag(s) for s in per_seed)}, chi2_ref mean={np.mean([s['chi2_ref'] for s in per_seed]):.2f})")
        for name, kw in VARIANTS:
            c = classify_row(per_seed, **kw)
            print(f"      {name:30s} {c['identified']:2d}/{c['not_identified']:2d}/{c['optimiser_failed']:2d} -> {c['cls']}")
save_results()

=== E4F: zonal h-only, class per U0 (identified/not-id/opt-failed -> class), chi2_ref = chi2 at the true zonal parameters (model-error indicator) ===
[rich]
   U0=0.15  same: 0/10/0 NOT_IDENTIFIED  chi2_ref=1.00 | indep: 0/10/0 NOT_IDENTIFIED  chi2_ref=1.04
   U0=0.3   same: 0/10/0 NOT_IDENTIFIED  chi2_ref=1.00 | indep: 0/10/0 NOT_IDENTIFIED  chi2_ref=1.03
   U0=0.5   same: 0/10/0 NOT_IDENTIFIED  chi2_ref=1.00 | indep: 0/10/0 NOT_IDENTIFIED  chi2_ref=1.03
   U0=0.7   same: 0/10/0 NOT_IDENTIFIED  chi2_ref=1.00 | indep: 2/8/0 MARGINAL        chi2_ref=1.03
   U0=1.0   same: 0/10/0 NOT_IDENTIFIED  chi2_ref=1.00 | indep: 2/8/0 MARGINAL        chi2_ref=1.03
   U0* per arm: {'same': None, 'indep': None}
[sparse4]
   U0=0.15  same: 0/10/0 NOT_IDENTIFIED  chi2_ref=1.00 | indep: 0/10/0 NOT_IDENTIFIED  chi2_ref=3.81
   U0=0.3   same: 0/10/0 NOT_IDENTIFIED  chi2_ref=1.00 | indep: 0/10/0 NOT_IDENTIFIED  chi2_ref=3.65
   U0=0.5   same: 0/10/0 NOT_IDENTIFIED  chi2_ref=1.00 | indep: 0/10/0 NOT_IDENTIF

## Cell 7 — E1F: free field (lambda path), both arms, h and huv (as v3 E1)

In [8]:
E1_CONFIGS = [(U0, reg, var) for U0 in CFG["U0_LIST"] for reg in REGIMES for var in ("h", "huv")]
if TEST_MODE: E1_CONFIGS = E1_CONFIGS[:2]
t0 = time.time()
if RUN["E1F"]:
    for arm in ARMS:
        use_arm(arm)
        done = {(r["U0"], r["regime"], r["variant"]) for r in RESULTS["arms"][arm]["E1F"]}
        for (U0, reg, var) in E1_CONFIGS:
            if (U0, reg, var) in done: continue
            P_use = CFG["P_OBS"] if var == "h" else CFG["P_OBS"] // 3
            pb = build_problem(U0, reg, var, P_use, noise_seed=0)
            rows, dp, orc, fields = fit_free_path(pb, CFG["L_E1"], CF_T)
            RESULTS["arms"][arm]["E1F"].append(dict(U0=U0, regime=reg, variant=var, n_obs=pb["n_obs"], rows=rows, dp=dp, oracle=orc,
                                                    skill_dp=rows[dp]["skill"], skill_oracle=rows[orc]["skill"], r_dp=rows[dp]["r"],
                                                    recovery=bool(rows[dp]["skill"] >= 0.5), oracle_interior=bool(0 < orc < len(rows) - 1)))
            save_results()
            print(f"[E1F:{arm}] U0={U0} {reg} {var}  skill_DP={rows[dp]['skill']:.2f} (lam {rows[dp]['lam']:.0f})  skill_ORC={rows[orc]['skill']:.2f} (lam {rows[orc]['lam']:.0f})  r_DP={rows[dp]['r']:.2f}  [{time.time()-t0:.0f}s]")
print("\n=== E1F summary: skill at the discrepancy-principle lambda (recovery := skill >= 0.5) and at the oracle lambda ===")
print(f"{'U0':>5} {'regime':8s} {'var':4s} | " + " | ".join(f"{a:>5}: skillDP recov skillORC" for a in ARMS))
for (U0, reg, var) in E1_CONFIGS:
    cells = []
    for a in ARMS:
        m = [r for r in RESULTS["arms"][a]["E1F"] if (r["U0"], r["regime"], r["variant"]) == (U0, reg, var)]
        cells.append(f"{m[0]['skill_dp']:8.2f} {str(m[0]['recovery']):>5} {m[0]['skill_oracle']:8.2f}" if m else "      (missing)       ")
    print(f"{U0:>5} {reg:8s} {var:4s} | " + " | ".join(f"      {c}" for c in cells))
save_results(); print("\nSaved ->", OUT_PATH, "| pre-registration hash:", PREREG_HASH)

[E1F:same] U0=0.15 rich h  skill_DP=-0.03 (lam 1000)  skill_ORC=0.01 (lam 1)  r_DP=0.90  [118s]
[E1F:same] U0=0.15 rich huv  skill_DP=0.47 (lam 30)  skill_ORC=0.68 (lam 1)  r_DP=0.94  [440s]
[E1F:same] U0=0.15 sparse4 h  skill_DP=-0.02 (lam 1000)  skill_ORC=0.35 (lam 3)  r_DP=0.87  [582s]
[E1F:same] U0=0.15 sparse4 huv  skill_DP=0.42 (lam 300)  skill_ORC=0.56 (lam 30)  r_DP=0.93  [874s]
[E1F:same] U0=1.0 rich h  skill_DP=-0.02 (lam 1000)  skill_ORC=0.57 (lam 1)  r_DP=0.92  [1013s]
[E1F:same] U0=1.0 rich huv  skill_DP=0.64 (lam 30)  skill_ORC=0.71 (lam 1)  r_DP=0.96  [1357s]
[E1F:same] U0=1.0 sparse4 h  skill_DP=0.57 (lam 100)  skill_ORC=0.68 (lam 10)  r_DP=0.93  [1477s]
[E1F:same] U0=1.0 sparse4 huv  skill_DP=0.63 (lam 300)  skill_ORC=0.69 (lam 10)  r_DP=0.95  [1820s]
[E1F:indep] U0=0.15 rich h  skill_DP=-0.09 (lam 1000)  skill_ORC=0.05 (lam 1)  r_DP=0.92  [1970s]
[E1F:indep] U0=0.15 rich huv  skill_DP=0.44 (lam 1)  skill_ORC=0.53 (lam 10)  r_DP=0.83  [2269s]
[E1F:indep] U0=0.15 sparse